# **Installation and Licensing**

In [ ]:
!pip install -q gamspy # Installs the GAMSPy package. GAMSPy documentation: https://gamspy.readthedocs.io/en/latest/

In [ ]:
!gamspy show license

In [ ]:
!gamspy list solvers

In [ ]:
!gamspy install license <path_to_ascii_file or access code> # paste your license here (if demo-license is not enough)

In [ ]:
# Optional: Install all solvers
!gamspy install solver --install-all-solvers

In [ ]:
!gamspy list solvers # Lists all available solvers in the gamspy package

# **Mathematical Model**

**Cardinality-Constrained Portfolio Optimization Problem**

\begin{align}
\min \quad & \sum_{i=1}^n \sum_{j=1}^n Q_{ij} x_i x_j \\
\text{subject to} \quad
& \sum_{i=1}^n x_i = 1 \quad && \text{(Budget constraint)} \\
& x_i \leq y_i, \quad \forall i = 1, \ldots, n \quad && \text{(Linking constraint)} \\
& \sum_{i=1}^n y_i \leq 5 \quad && \text{(Cardinality constraint)} \\
& x_i \geq 0, \quad \forall i = 1, \ldots, n \quad && \text{(Non-negativity)} \\
& y_i \in \{0, 1\}, \quad \forall i = 1, \ldots, n \quad && \text{(Binary selection variables)}
\end{align}

$$
\begin{array}{l l l r r r r}
\text{Name} & \text{Type} & \text{C} & \#\text{Vars} & \#\text{BinVars} & \#\text{IntVars} & \#\text{Cons} \\
\hline
CCPOP5\_20 & MBQP & Convex & 40 & 20 & 0 & 22 \\
CCPOP5\_50 & MBQP & Convex & 100 & 50 & 0 & 52 \\
CCPOP5\_100 & MBQP & Convex & 200 & 100 & 0 & 102 \\
CCPOP5\_186 & MBQP & Convex & 372 & 186 & 0 & 188 \\
CCPOP5\_219 & MBQP & Convex & 438 & 219 & 0 & 221 \\  
CCPOP5\_300 & MBQP & Convex & 600 & 300 & 0 & 302 \\
\end{array}
$$

#  **Solver Setup**


## CCPOP5_20

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data & covariance
tickers = [
    "ADBE", "CMCSA", "PEP", "NFLX", "TXN", "QCOM", "AVGO", "AMD", "SBUX", "GILD",
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA", "PYPL", "INTC", "CSCO"
] # 20
data = yf.download(tickers, start='2022-01-01', end='2025-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values

# Create GAMSPy container
m = Container()

# Sets and Parameters
assets = [f"asset_{i+1}" for i in range(len(cov))]

# Sets
i = Set(m, name="i", records=[f"asset_{k+1}" for k in range(len(cov))])
ii = Alias(m, name="ii", alias_with=i)  # alias for second index

Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(len(cov)) for c in range(len(cov))])

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")  # investment weights
y = Variable(m, name="y", domain=[i], type="Binary")    # selection flags

# Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 3 assets

# Objective (minimize portfolio variance)
obj = Sum([i, ii], Q[i, ii] * x[i] * x[ii])

# Model
model = Model(m, name="CCPOP5_20",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MIN,
              objective=obj)

# Solve with SHOT
import sys

model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver":2})

In [ ]:
# Convert GAMSPy model to GAMS (Note: One needs to download both the .gms and .gdx file)
model.toGams(path="gams")

After this, the files were opened and combined in GAMS Studio by GAMS Convert. Below is the rest of the codes for this type of problem.


## CCPOP5_50

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data & covariance
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA", "PYPL", "INTC", "CSCO",
    "ADBE", "CMCSA", "PEP", "NFLX", "TXN", "QCOM", "AVGO", "AMD", "SBUX", "GILD",
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "PG", "T", "VZ", "WMT", "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS"
] # 50
data = yf.download(tickers, start='2022-01-01', end='2025-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values

# Create GAMSPy container
m = Container()

# Sets and Parameters
assets = [f"asset_{i+1}" for i in range(len(cov))]

# Sets
i = Set(m, name="i", records=[f"asset_{k+1}" for k in range(len(cov))])
ii = Alias(m, name="ii", alias_with=i)  # alias for second index

Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(len(cov)) for c in range(len(cov))])

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")  # investment weights
y = Variable(m, name="y", domain=[i], type="Binary")    # selection flags

# Constraints
budget = Equation(m, name="budget")
budget[:] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 5 assets

# Objective (minimize portfolio variance)
obj = Sum([i, ii], Q[i, ii] * x[i] * x[ii])

# Model
model = Model(m, name="CCPOP5_50",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MIN,
              objective=obj)

# Solve with SHOT
import sys

model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver":2})


## CCPOP5_100


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data & covariance
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA", "PYPL", "INTC", "CSCO",
    "ADBE", "CMCSA", "PEP", "NFLX", "TXN", "QCOM", "AVGO", "AMD", "SBUX", "GILD",
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "PG", "T", "VZ", "WMT", "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "BKNG", "COST", "DE", "DHR", "EL", "FDX", "GE", "GM",
    "HON", "IBM", "ISRG", "JNJ", "LMT", "LIN", "LRCX", "MAR", "MDLZ", "MET",
    "MO", "MS", "NEE", "NOC", "NVDA", "PYPL", "SBUX", "SCHW", "SO", "SPGI",
    "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO", "VZ", "WBA", "WFC",
    "XEL", "ZBRA", "ZBH", "SNAP", "DOCU", "UBER", "ZM", "SHOP",
     "ADP", "BMY", "C",  "DOW",  "EMR",  "F",
    "GM",  "HCA",   "LVS",  "PYPL", "BK"
]# 100
data = yf.download(tickers, start='2022-01-01', end='2024-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values

# Create GAMSPy container
m = Container()

# Sets and Parameters
assets = [f"asset_{i+1}" for i in range(len(cov))]

# Sets
i = Set(m, name="i", records=[f"asset_{k+1}" for k in range(len(cov))])
ii = Alias(m, name="ii", alias_with=i)  # alias for second index

Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(len(cov)) for c in range(len(cov))])

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")  # investment weights
y = Variable(m, name="y", domain=[i], type="Binary")    # selection flags

# Constraints
budget = Equation(m, name="budget")
budget[:] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # 5 assets

# Objective (minimize portfolio variance)
obj = Sum([i, ii], Q[i, ii] * x[i] * x[ii])

# Model
model = Model(m, name="CCPOP_5_100",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MIN,
              objective=obj)

# Solve with SHOT
import sys

model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver":2})

## CCPOP5_186

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data & covariance
tickers = tickers = [
     "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "DHR", "EL", "FDX", "GE", "GM",
    "IBM", "ISRG", "LMT", "LIN", "LRCX", "MAR", "MDLZ", "MET",
    "MO", "MS", "NEE", "NOC", "NVDA", "PYPL", "SBUX", "SCHW", "SO", "SPGI",
    "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO", "WFC",
     "ADP", "BMY", "C",  "DOW",  "EMR",  "F",
    "GM",  "HCA",   "LVS",  "PYPL", "BK",
    "SHW", "EOG", "TRV", "FDX", "HUM",
    "EBAY", "DHI", "MCK", "COF", "KR", "WBD", "ED",
    "DVN", "SBAC", "OTIS", "EFX", "CHTR", "BKR", "KEYS", "WMB", "VRSK",
    "FTV", "STT", "PPG", "CEG",  "CTVA",
    "PAYX", "BIIB", "XEL", "LHX", "MTD", "DRI", "ZBH", "KLAC", "TSCO", "TDG",
    "RMD", "PCG", "DOV", "VICI", "CBRE", "CMS", "EXC", "DLTR",
    "ATO", "CDW", "HIG", "LDOS", "CNP", "MLM", "LRCX", "AEE", "EIX",
    "RCL", "FANG", "PPL", "FAST", "IFF", "COR", "ETR", "TRGP", "MAA",
    "AKAM", "TSN", "EPAM", "NUE", "WEC", "HPE", "IR", "AIZ", "MKC", "RSG",
    "CAH", "VTR", "CHD", "BALL", "NDAQ", "NI", "AVY", "CRL", "DTE",
    "SJM", "FE", "WAT", "TTWO", "EXR", "BBY", "HOLX", "SWKS", "JKHY", "LKQ",
    "FMC", "TER", "NRG", "TYL", "PHM", "JBHT",
    "RL", "BR", "ZBRA", "BXP", "GNRC", "VMC", "STE", "ESS", "APA",
    "INCY", "TECH", "PKG", "CF", "HBAN", "AFL", "NVR", "ALB", "TXT", "DXCM",
    "KIM", "UHS", "BWA", "MOS", "BEN", "IP", "NDSN", "HII", "ARE", "PWR",

    "LKQ", "IPG", "QRVO", "WY", "GNW", "TXT", "PFG"
]

data = yf.download(tickers, start='2022-01-01', end='2025-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values

# Create GAMSPy container
m = Container()

# Sets and Parameters
assets = [f"asset_{i+1}" for i in range(len(cov))]

# Sets
i = Set(m, name="i", records=[f"asset_{k+1}" for k in range(len(cov))])
ii = Alias(m, name="ii", alias_with=i)  # alias for second index

Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(len(cov)) for c in range(len(cov))])

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")  # investment weights
y = Variable(m, name="y", domain=[i], type="Binary")    # selection flags

# Constraints
budget = Equation(m, name="budget")
budget[:] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 3 assets

# Objective (minimize portfolio variance)
obj = Sum([i, ii], Q[i, ii] * x[i] * x[ii])

# Model
model = Model(m, name="CCPOP5_186",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MIN,
              objective=obj)

# Solve with SHOT
import sys

model.solve(output=sys.stdout, solver="convert")
#model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver":2})


## CCPOP5_219


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data & covariance
tickers = tickers = [
     "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "DHR", "EL", "FDX", "GE", "GM",
    "IBM", "ISRG", "LMT", "LIN", "LRCX", "MAR", "MDLZ", "MET",
    "MO", "MS", "NEE", "NOC", "NVDA", "PYPL", "SBUX", "SCHW", "SO", "SPGI",
    "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO", "WFC",
     "ADP", "BMY", "C",  "DOW",  "EMR",  "F",
    "GM",  "HCA",   "LVS",  "PYPL", "BK",
    "AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "META", "AVGO", "TSLA",
    "XOM", "UNH", "LLY", "PG", "MA", "HD", "MRK", "CVX",
    "PEP", "ABBV", "COST", "BAC", "ADBE", "KO", "WMT", "CRM", "NFLX", "TMO",
    "ABT", "ACN", "GS", "SPGI", "CVS", "SBUX", "BLK", "MDT",
    "ISRG", "LMT", "TGT", "BKNG", "GILD",
    "MMC", "ADP", "VRTX", "CB", "ELV", "REGN", "PNC", "FI", "C", "ADI",
    "CSX", "MDLZ", "SCHW", "PGR", "MU", "DUK", "EQIX", "SO", "HCA",
    "BDX", "ICE", "CL", "ITW", "APD", "SHW", "EOG", "TRV", "FDX", "HUM",
    "EBAY", "DHI", "MCK", "COF", "KR", "WBD", "ED",
    "DVN", "SBAC", "OTIS", "EFX", "CHTR", "BKR", "KEYS", "WMB", "VRSK",
    "FTV", "STT", "PPG", "CEG",  "CTVA",
    "PAYX", "BIIB", "XEL", "LHX", "MTD", "DRI", "ZBH", "KLAC", "TSCO", "TDG",
    "RMD", "PCG", "DOV", "VICI", "CBRE", "CMS", "EXC", "DLTR",
    "ATO", "CDW", "HIG", "LDOS", "CNP", "MLM", "LRCX", "AEE", "EIX",
    "RCL", "FANG", "PPL", "FAST", "IFF", "COR", "ETR", "TRGP", "MAA",
    "AKAM", "TSN", "EPAM", "NUE", "WEC", "HPE", "IR", "AIZ", "MKC", "RSG",
    "CAH", "VTR", "CHD", "BALL", "NDAQ", "NI", "AVY", "CRL", "DTE",
    "SJM", "FE", "WAT", "TTWO", "EXR", "BBY", "HOLX", "SWKS", "JKHY", "LKQ",
    "FMC", "TER", "NRG", "TYL", "PHM", "JBHT",
    "RL", "BR", "ZBRA", "BXP", "GNRC", "VMC", "STE", "ESS", "APA",
    "INCY", "TECH", "PKG", "CF", "HBAN", "AFL", "NVR", "ALB", "TXT", "DXCM",
    "KIM", "UHS", "BWA", "MOS", "BEN"
]

data = yf.download(tickers, start='2022-01-01', end='2025-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values

# Create GAMSPy container
m = Container()

# Sets and Parameters
assets = [f"asset_{i+1}" for i in range(len(cov))]

# Sets
i = Set(m, name="i", records=[f"asset_{k+1}" for k in range(len(cov))])
ii = Alias(m, name="ii", alias_with=i)  # alias for second index

Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(len(cov)) for c in range(len(cov))])

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")  # investment weights
y = Variable(m, name="y", domain=[i], type="Binary")    # selection flags

# Constraints
budget = Equation(m, name="budget")
budget[:] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 3 assets

# Objective (minimize portfolio variance)
obj = Sum([i, ii], Q[i, ii] * x[i] * x[ii])

# Model
model = Model(m, name="CCPOP5_219",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MIN,
              objective=obj)

# Solve with SHOT
import sys


model.solve(output=sys.stdout, solver="convert")
#model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver":2})


## CCPOP5_300


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data & covariance
tickers = tickers = [
     "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "DHR", "EL", "FDX", "GE", "GM",
    "IBM", "ISRG", "LMT", "LIN", "LRCX", "MAR", "MDLZ", "MET",
    "MO", "MS", "NEE", "NOC", "NVDA", "PYPL", "SBUX", "SCHW", "SO", "SPGI",
    "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO", "WBA", "WFC",
    "XEL", "ZBRA", "ZBH", "SNAP", "DOCU", "UBER", "ZM", "SHOP",
     "ADP", "BMY", "C",  "DOW",  "EMR",  "F",
    "GM",  "HCA",   "LVS",  "PYPL", "BK",
    "AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "META", "AVGO", "TSLA",
    "XOM", "UNH", "LLY", "PG", "MA", "HD", "MRK", "CVX",
    "PEP", "ABBV", "COST", "BAC", "ADBE", "KO", "WMT", "CRM", "NFLX", "TMO",
    "ABT", "ACN", "PFE", "CSCO", "MCD", "DHR", "LIN", "INTC", "QCOM", "NEE",
    "WFC", "TXN", "PM", "MS", "BMY", "AMGN", "ORCL", "UNP", "AVB",
    "LOW", "UPS", "RTX", "AMD", "GS", "SPGI", "CVS", "SBUX", "BLK", "MDT",
    "ISRG", "LMT", "TGT", "BKNG", "GILD",
    "MMC", "ADP", "VRTX", "CB", "ELV", "REGN", "PNC", "FI", "C", "ADI",
    "CSX", "MDLZ", "SCHW", "PGR", "MU", "DUK", "EQIX", "SO", "HCA",
    "BDX", "ICE", "CL", "ITW", "APD", "SHW", "EOG", "TRV", "FDX", "HUM",
    "NKE", "PSA", "EMR", "ETN", "TFC", "GM", "AON", "WM", "ROP", "AIG",
    "MAR", "USB", "MCO", "D", "EW", "IDXX", "AEP", "FTNT", "ADSK",
    "ROST", "MNST", "KDP", "CMG", "MET", "CTAS", "ORLY", "PCAR", "CTSH", "AZO",
    "SPG", "WTW", "CME", "DLR", "PH", "F", "VLO", "PRU", "OXY", "MSI",
    "ALL", "TEL", "HLT", "HSY", "PEG",
    "EBAY", "DHI", "HES", "MCK", "COF", "KR", "WBD", "ED",
    "DVN", "SBAC", "OTIS", "EFX", "CHTR", "BKR", "DFS", "KEYS", "WMB", "VRSK",
    "FTV", "STT", "PPG", "CEG",  "CTVA",
    "PAYX", "BIIB", "XEL", "LHX", "MTD", "DRI", "ZBH", "KLAC", "TSCO", "TDG",
    "RMD", "PCG", "DOV", "VICI", "CBRE", "CMS", "EXC", "DLTR",
    "ATO", "CDW", "HIG", "LDOS", "CNP", "MLM", "LRCX", "AEE", "EIX",
    "RCL", "FANG", "PPL", "FAST", "IFF", "COR", "ETR", "TRGP", "MAA",
    "AKAM", "TSN", "EPAM", "NUE", "WEC", "HPE", "IR", "AIZ", "MKC", "RSG",
    "CAH", "VTR", "CHD", "BALL", "NDAQ", "NI", "AVY", "CRL", "DTE",
    "SJM", "FE", "WAT", "TTWO", "EXR", "BBY", "HOLX", "SWKS", "JKHY", "LKQ",
    "FMC", "TER", "NRG", "TYL", "PHM", "JBHT",
    "RL", "BR", "ZBRA", "BXP", "GNRC", "VMC", "STE", "ESS", "APA",
    "INCY", "TECH", "PKG", "CF", "HBAN", "AFL", "NVR", "ALB", "TXT", "DXCM",
    "KIM", "UHS", "BWA", "MOS", "BEN", "IP", "NDSN", "HII", "ARE", "PWR",
    "WHR", "LNT", "EMN", "TAP", "AMCR", "XRAY", "KMX", "L", "CINF", "SNA",
    "FRT", "NOV", "CAG", "GRMN", "WRB", "PARA", "ALLE", "AOS",
    "LKQ", "IPG", "QRVO", "WY", "GNW", "TXT", "PFG"
] # 300

data = yf.download(tickers, start='2022-01-01', end='2025-05-31')['Close']
returns = data.pct_change().dropna()
cov = returns.cov().values

# Create GAMSPy container
m = Container()

# Sets and Parameters
assets = [f"asset_{i+1}" for i in range(len(cov))]

# Sets
i = Set(m, name="i", records=[f"asset_{k+1}" for k in range(len(cov))])
ii = Alias(m, name="ii", alias_with=i)  # alias for second index

Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(len(cov)) for c in range(len(cov))])

# Variables
x = Variable(m, name="x", domain=[i], type="Positive")  # investment weights
y = Variable(m, name="y", domain=[i], type="Binary")    # selection flags

# Constraints
budget = Equation(m, name="budget")
budget[:] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 3 assets

# Objective (minimize portfolio variance)
obj = Sum([i, ii], Q[i, ii] * x[i] * x[ii])

# Model
model = Model(m, name="CCPOP5_300",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MIN,
              objective=obj)

# Solve with SHOT
import sys

model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver":2})
